In [ ]:
# Generalized Assignment Problem (Cloud HPC Scheduling)
# ------------------------------------------------------
# Jobs are assigned to GPU clusters under capacity constraints.
# Costs reflect queue delay, job priority, and resource intensity.

from optilab.aliases import Model, GRB, quicksum
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem data (HPC / cloud system)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(11)

n_clusters = rng.integers(4, 7)   # GPU clusters (agents)
n_jobs = rng.integers(8, 15)      # ML training jobs (tasks)

# -------------------------------------------------------------------------------
# Cluster capacities (GPU-hours per scheduling window)
# -------------------------------------------------------------------------------
capacity = rng.integers(80, 140, size=n_clusters)

# -------------------------------------------------------------------------------
# Job resource demand (GPU-hours required)
# -------------------------------------------------------------------------------
gpu_demand = rng.integers(10, 50, size=n_jobs)

# -------------------------------------------------------------------------------
# Priority levels (higher = more important)
# -------------------------------------------------------------------------------
priority = rng.integers(1, 6, size=n_jobs)

# -------------------------------------------------------------------------------
# Feasibility design (important)
# Ensure total demand is safely within total capacity by scaling down demand
# -------------------------------------------------------------------------------
scale = 0.8 * capacity.sum() / gpu_demand.sum()
gpu_demand = np.maximum(1, (gpu_demand * scale).astype(int))

# -------------------------------------------------------------------------------
# Cost structure: "queue-aware scheduling penalty"
# -------------------------------------------------------------------------------
# Interpretation:
# - higher cluster load cost → congestion penalty
# - higher job priority → lower cost (negative weighting effect)
# - higher resource demand → more expensive to place

base_latency = rng.integers(1, 20, size=(n_clusters, n_jobs))

cost = np.zeros((n_clusters, n_jobs))

for i in range(n_clusters):
    for j in range(n_jobs):
        cost[i, j] = (
            base_latency[i, j]
            + 0.05 * gpu_demand[j]
            - 0.8 * priority[j]
        )

cost = cost - cost.min() + 1  # shift to keep positive

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
# x[i][j] = 1 if job j is assigned to cluster i
x = [
    [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n_jobs)]
    for i in range(n_clusters)
]

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# Each job must be assigned to exactly one cluster
for j in range(n_jobs):
    m.add_constraint(
        quicksum(x[i][j] for i in range(n_clusters)) == 1
    )

# Cluster capacity constraints (generalized assignment structure)
for i in range(n_clusters):
    m.add_constraint(
        quicksum(gpu_demand[j] * x[i][j] for j in range(n_jobs))
        <= capacity[i]
    )

# -------------------------------------------------------------------------------
# Objective: minimize scheduling penalty
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(
        cost[i][j] * x[i][j]
        for i in range(n_clusters)
        for j in range(n_jobs)
    ),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
assignment = np.zeros((n_clusters, n_jobs))

for i in range(n_clusters):
    for j in range(n_jobs):
        assignment[i, j] = x[i][j].x

# -------------------------------------------------------------------------------
# Diagnostics
# -------------------------------------------------------------------------------
cluster_load = assignment @ gpu_demand

print(f"Backend:        {m.backend_name}")
print(f"Clusters:       {n_clusters}")
print(f"Jobs:           {n_jobs}")
print(f"Total capacity: {capacity.sum()}")
print(f"Total demand:   {gpu_demand.sum()}")
print()
print("Cost matrix (scheduling penalty):\n", cost)
print()
print("Assignment matrix:\n", assignment)
print(f"Cluster loads:  {cluster_load}")
print(f"Cluster cap.s:  {capacity}")
print(f"Total cost:     {float(np.sum(assignment * cost))}")

# -------------------------------------------------------------------------------
# Visualization
# -------------------------------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.imshow(assignment, cmap="Greens")
plt.title("Generalized Assignment: Jobs → GPU Clusters")
plt.xlabel("Jobs")
plt.ylabel("Clusters")
plt.colorbar(label="Assignment")
plt.show()